# BCI Motor Imagery Classification — EEG Preprocessing Pipeline

This notebook implements the complete, end-to-end preprocessing pipeline for EEG Motor Imagery (Left vs Right Hand) classification as specified in the project guidelines.

### Pipeline Workflow:
1. **Data Ingestion & Calibration**: Standardize channels (, , , ), align timestamp jitter onto uniform  = 250	ext{ Hz}$ grid, calibrate ADC counts to $\mu	ext{V}$.
2. **Drift Removal & Notch Filtering**: Linear detrending and 0	ext{ Hz}$ IIR notch filter (=30$).
3. **Zero-Phase Butterworth Bandpass**: bash.5 - 40	ext{ Hz}$ (broadband) and  - 30	ext{ Hz}$ (sensorimotor $\mu/eta$ band).
4. **Spatial Re-referencing**: Common Average Reference (CAR) to suppress global environmental noise.
5. **Artifact Rejection & Audit**: Peak-to-peak amplitude thresholding, flatline/dead channel detection, and statistical robust variance hBcscore filtering.
6. **Baseline Correction & Epoching**: Pre-stimulus baseline (bash.0 - 0.5	ext{ s}$) subtraction, task epoch (bash.5 - 3.5	ext{ s}$) slicing.
7. **Outlier Conditioning & Normalization**: Statistical Winsorization (.5\sigma$) and per-channel hBcscore standardization.
8. **Deliverables Export**:  arrays for deep learning (EEGNet) and classical ML (CSP+SVM), serialized pipeline , and audit CSV.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal

from config import PreprocessingConfig, DEFAULT_CONFIG
from preprocessing.loader import load_eeg_dataset, load_single_trial
from preprocessing.pipeline import EEGPreprocessingPipeline
from preprocessing.filters import butter_bandpass_filter

sns.set_theme(style="whitegrid", font_scale=1.1)
print("Environment initialized.")

In [ ]:
# Step 1: Load and calibrate raw dataset
config = DEFAULT_CONFIG
signals, labels, filenames, time_axis, meta_df = load_eeg_dataset(
    data_dir="raw_data",
    labels_path="labels.csv",
    config=config
)
print(f"Loaded {len(signals)} trials with shape {signals.shape}")
print(f"Class Distribution:
{pd.Series(labels).value_counts()}")

In [ ]:
# Step 2 to 8: Run Full Preprocessing Pipeline
pipeline = EEGPreprocessingPipeline(config=config)
results = pipeline.process_batch(
    signals=signals,
    time_axis=time_axis,
    labels=labels,
    filenames=filenames
)

X_clean = results["X_clean"]
y_clean = results["y_clean"]
epoch_time = results["epoch_time_axis"]
report_df = results["report_df"]

print(f"Clean Trials Retained: {len(X_clean)} / {len(signals)}")
print(f"Clean Shape: {X_clean.shape} (trials x channels x time_samples)")

In [ ]:
# Inspect Artifact Audit Summary
print("Top Rejection Reasons:")
print(report_df["rejection_reasons"].value_counts().head(10))

In [ ]:
# Verify Preprocessed Arrays Saved
print("Processed Data Directory Contents:")
for f in sorted(os.listdir("processed_data")):
    print(f" - {f} ({os.path.getsize(os.path.join('processed_data', f))/1024:.1f} KB)")